In [ ]:
import os
print(os.getcwd())

In [ ]:
ROOT_DIR = r"D:\jupyter_workspace\brain_tumor_detection\dataset"
number_of_images = {}

for dir in os.listdir(ROOT_DIR):
    number_of_images[dir] = os.listdir(os.path.join(ROOT_DIR, dir))

In [ ]:
number_of_images

In [ ]:
import os
import shutil
import random

ROOT_DIR = r"D:\jupyter_workspace\brain_tumor_detection\dataset"
TRAIN_DIR = os.path.join(ROOT_DIR, "Training")
VAL_DIR = os.path.join(ROOT_DIR, "Validation")

# Create Validation folder if not exists
if not os.path.exists(VAL_DIR):
    os.makedirs(VAL_DIR)

split_ratio = 0.2  # 20% validation

for class_name in os.listdir(TRAIN_DIR):
    class_train_path = os.path.join(TRAIN_DIR, class_name)
    class_val_path = os.path.join(VAL_DIR, class_name)

    if not os.path.exists(class_val_path):
        os.makedirs(class_val_path)

    images = os.listdir(class_train_path)
    random.shuffle(images)

    val_count = int(len(images) * split_ratio)
    val_images = images[:val_count]

    for img in val_images:
        src = os.path.join(class_train_path, img)
        dst = os.path.join(class_val_path, img)
        shutil.move(src, dst)  # change to shutil.copy if you prefer copy

print("Validation folder created successfully.")

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dropout, Flatten, Dense, BatchNormalization, GlobalAvgPool2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras
import keras
%matplotlib inline

In [ ]:
model = Sequential()

model.add(Conv2D(filters=16, kernel_size=(3,3), activation="relu", input_shape=(224, 224, 3)))

model.add(Conv2D(filters=32, kernel_size=(3,3), activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Conv2D(filters=64, kernel_size=(3,3), activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Conv2D(filters=128, kernel_size=(3,3), activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Dropout(rate=0.25))

model.add(Flatten())
model.add(Dense(units=64, activation="relu"))

model.add(Dropout(rate=0.25))

model.add(Dense(units=4, activation='softmax'))

model.summary()

In [ ]:
model.compile(
    optimizer='adam', 
    loss=keras.losses.categorical_crossentropy,
    metrics=['accuracy']
)

In [ ]:
def preprocessing_images1(path):
    image_data = ImageDataGenerator(zoom_range=0.2, shear_range=0.2, rescale=1/255, horizontal_flip=True)
    image = image_data.flow_from_directory(directory = path, target_size=(224, 224), batch_size=32, class_mode='binary')

    return image

In [ ]:
path = "dataset/Training"
training_data = preprocessing_images1(path)

In [ ]:
def preprocessing_images2(path):
    image_data = ImageDataGenerator(rescale=1/255)
    image = image_data.flow_from_directory(directory = path, target_size=(224, 224), batch_size=32, class_mode='categorical')

    return image

In [ ]:
path = "dataset/Testing"
test_data = preprocessing_images2(path)

In [ ]:
path = "dataset/Validation"
val_data = preprocessing_images2(path)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

es = EarlyStopping(monitor="val_accuracy", min_delta=0.01, patience=3, verbose=1, mode='auto')
mc = ModelCheckpoint(monitor="val_accuracy", filepath="./best_model.keras", verbose=1, save_best_only=True, mode="auto")

cd = [es, mc]

In [ ]:
import scipy
hs = model.fit(
    x=training_data,
    epochs=30, 
    verbose=1,
    validation_data=val_data,
    callbacks=cd
)

In [ ]:
h = hs.history
h.keys()

In [ ]:
from tensorflow.keras.models import load_model
model = load_model("best_model.keras")

In [ ]:
loss, accuracy = model.evaluate(test_data)
print(f"accuracy is: {accuracy}")

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

In [ ]:
path = r"dataset/Testing/notumor/Te-no_7.jpg"
img = load_img(path, target_size=(224, 224))

input_arr = img_to_array(img) / 255
input_arr.shape

input_arr = np.expand_dims(input_arr, axis=0)

pred = model.predict(input_arr)[0]
pred

index = np.argmax(pred)
mp = {0: 'glioma', 1: 'meningioma', 2: 'notumor', 3: 'pituitary'}

mp[index]

In [ ]:
training_data.class_indices

In [ ]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import Model

In [ ]:
def preprocessing_images3(path):
    image_data = ImageDataGenerator(zoom_range=0.2, shear_range=0.2, preprocessing_function=preprocess_input, horizontal_flip=True)
    image = image_data.flow_from_directory(directory = path, target_size=(224, 224), batch_size=32, class_mode='categorical')

    return image

In [ ]:
def preprocessing_images4(path):
    image_data = ImageDataGenerator(preprocessing_function=preprocess_input)
    image = image_data.flow_from_directory(directory = path, target_size=(224, 224), batch_size=32, class_mode='categorical')

    return image

In [ ]:
path = "dataset/Training"
training_data = preprocessing_images3(path)

In [ ]:
path = "dataset/Testing"
test_data = preprocessing_images4(path)

In [ ]:
base_model = MobileNet(input_shape=(224, 224, 3), include_top=False)

In [ ]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

In [ ]:
base_model.summary()

In [ ]:
X = Flatten()(base_model.output)
X = Dense(units=4, activation="softmax")(X)

model = Model(base_model.input, X) 

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='rmsprop', loss=keras.losses.categorical_crossentropy, metrics=['accuracy'])

In [ ]:
es = EarlyStopping(monitor="val_accuracy", min_delta=0.01, patience=3, verbose=1, mode='auto')
mc = ModelCheckpoint(monitor="val_accuracy", filepath="./best_model.keras", verbose=1, save_best_only=True, mode="auto")

cb = [es, mc]

In [ ]:
hs = model.fit(
    training_data,
    epochs=30,
    validation_data=val_data,
    callbacks=cb
)

In [ ]:
model = load_model("best_model.keras")

In [ ]:
loss, accuracy = model.evaluate(test_data)
accuracy